# 00 Preprocessing — Bike Flow Data
Run this notebook **once** to merge all 65 monthly CSV files, aggregate to hourly, and save as a compact Parquet file.
All other notebooks load from `data/processed/bike_hourly.parquet` instead of the raw CSVs.

## 1. Load & Merge All Monthly CSVs

In [1]:
import pandas as pd
import glob
import os

column_names = [
    'sensor_id', 'direction', 'vehicle_type',
    'start_time', 'end_time', 'count'
]

file_paths = sorted(glob.glob('data/raw/sensor data/data-*.csv'))
print(f'Found {len(file_paths)} CSV files')
print(f'Date range: {file_paths[0][-11:-4]} → {file_paths[-1][-11:-4]}')

Found 65 CSV files
Date range: 2019-08 → 2024-12


In [2]:
# Load all files, filter FIETSERS, drop unnecessary columns
# Process one file at a time to avoid memory overload
chunks = []

for i, f in enumerate(file_paths):
    df = pd.read_csv(f, header=None, names=column_names)
    df = df[df['vehicle_type'] == 'FIETSERS']
    df = df.drop(columns=['vehicle_type', 'end_time'])
    df['sensor_id'] = df['sensor_id'].astype('int32')
    df['count'] = pd.to_numeric(df['count'], errors='coerce').astype('float32')
    df['start_time'] = pd.to_datetime(df['start_time'])
    chunks.append(df)

    if (i + 1) % 10 == 0:
        print(f'Loaded {i+1}/{len(file_paths)} files...')

bike_data = pd.concat(chunks, ignore_index=True)
del chunks

print(f'\nTotal records (15-min): {len(bike_data):,}')
print(f'Memory usage: {bike_data.memory_usage(deep=True).sum() / 1024**3:.2f} GB')
bike_data.head()

Loaded 10/65 files...
Loaded 20/65 files...
Loaded 30/65 files...
Loaded 40/65 files...
Loaded 50/65 files...
Loaded 60/65 files...

Total records (15-min): 29,429,672
Memory usage: 2.07 GB


,sensor_id,direction,start_time,count
0,1,IN,2019-08-01 00:00:00,0.0
1,1,IN,2019-08-01 00:15:00,0.0
2,1,IN,2019-08-01 00:30:00,0.0
3,1,IN,2019-08-01 00:45:00,0.0
4,1,IN,2019-08-01 01:00:00,1.0


## 2. Data Quality Checks

In [3]:
# Missing values
print('Missing values:')
print(bike_data.isnull().sum())

# Date range
print(f'\nDate range: {bike_data["start_time"].min()} → {bike_data["start_time"].max()}')

# Sensors
print(f'Unique sensors: {bike_data["sensor_id"].nunique()}')

# Extreme values (possible sensor errors)
p999 = bike_data['count'].quantile(0.999)
extreme = bike_data[bike_data['count'] > p999]
print(f'\nExtreme count values (> 99.9th percentile = {p999:.0f}): {len(extreme):,} records')
print(f'Max count value: {bike_data["count"].max():.0f}')

Missing values:
sensor_id          0
direction          0
start_time         0
count         777345
dtype: int64

Date range: 2019-08-01 00:00:00 → 2024-12-31 23:45:00
Unique sensors: 142

Extreme count values (> 99.9th percentile = 76): 27,955 records
Max count value: 5402


In [4]:
# Remove missing counts
before = len(bike_data)
bike_data = bike_data.dropna(subset=['count'])
print(f'Dropped {before - len(bike_data):,} rows with missing count')

# Cap extreme values at 99.9th percentile (likely sensor errors)
p999 = bike_data['count'].quantile(0.999)
n_capped = (bike_data['count'] > p999).sum()
bike_data['count'] = bike_data['count'].clip(upper=p999)
print(f'Capped {n_capped:,} extreme values at {p999:.0f}')

Dropped 777,345 rows with missing count
Capped 27,955 extreme values at 76


## 3. Add Time Features

In [5]:
bike_data['hour'] = bike_data['start_time'].dt.hour
bike_data['day_of_week'] = bike_data['start_time'].dt.day_name()
bike_data['month'] = bike_data['start_time'].dt.month
bike_data['year'] = bike_data['start_time'].dt.year

print('Time features added:')
print(bike_data[['hour', 'day_of_week', 'month', 'year']].head())

Time features added:
   hour day_of_week  month  year
0     0    Thursday      8  2019
1     0    Thursday      8  2019
2     0    Thursday      8  2019
3     0    Thursday      8  2019
4     1    Thursday      8  2019


## 4. Aggregate to Hourly

In [6]:
# Aggregate 15-min records to hourly
# This reduces ~29M rows to ~2-3M rows
bike_data['hour_timestamp'] = bike_data['start_time'].dt.floor('H')

bike_hourly = (
    bike_data.groupby(['sensor_id', 'hour_timestamp', 'direction',
                       'hour', 'day_of_week', 'month', 'year'])
    .agg(count=('count', 'sum'))
    .reset_index()
)

print(f'15-min records: {len(bike_data):,}')
print(f'Hourly records: {len(bike_hourly):,}')
print(f'Compression ratio: {len(bike_data)/len(bike_hourly):.1f}x')
print(f'Memory: {bike_hourly.memory_usage(deep=True).sum() / 1024**2:.0f} MB')
bike_hourly.head()

/var/folders/hc/1tjpx7j56wx800gc7kmd3wr80000gn/T/ipykernel_54830/45995554.py:3: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  bike_data['hour_timestamp'] = bike_data['start_time'].dt.floor('H')


15-min records: 28,652,327
Hourly records: 7,166,679
Compression ratio: 4.0x
Memory: 1036 MB


,sensor_id,hour_timestamp,direction,hour,day_of_week,month,year,count
0,1,2019-08-01 00:00:00,IN,0,Thursday,8,2019,0.0
1,1,2019-08-01 00:00:00,OUT,0,Thursday,8,2019,2.0
2,1,2019-08-01 01:00:00,IN,1,Thursday,8,2019,1.0
3,1,2019-08-01 01:00:00,OUT,1,Thursday,8,2019,0.0
4,1,2019-08-01 02:00:00,IN,2,Thursday,8,2019,1.0


## 6. Save Processed Files

In [8]:
bike_hourly['sensor_id'] = bike_hourly['sensor_id'].astype('int16')  # max = 142
bike_hourly['hour'] = bike_hourly['hour'].astype('int8')              # 0-23
bike_hourly['month'] = bike_hourly['month'].astype('int8')            # 1-12
bike_hourly['year'] = bike_hourly['year'].astype('int16')             # 2019-2024
bike_hourly['count'] = bike_hourly['count'].astype('float32')         # already is 

In [9]:
import os
os.makedirs('data/processed', exist_ok=True)

bike_hourly.to_parquet('data/processed/bike_hourly.parquet', index=False)

size = os.path.getsize('data/processed/bike_hourly.parquet') / 1024**2
print(f'Saved: data/processed/bike_hourly.parquet ({size:.1f} MB)')

Saved: data/processed/bike_hourly.parquet (20.7 MB)


## 7. Verify Output

In [10]:
# Quick sanity check on saved file
test = pd.read_parquet('data/processed/bike_hourly.parquet')
print(f'Rows: {len(test):,}')
print(f'Columns: {test.columns.tolist()}')
print(f'Date range: {test["hour_timestamp"].min()} → {test["hour_timestamp"].max()}')
print(f'Sensors: {test["sensor_id"].nunique()}')
print(f'Memory: {test.memory_usage(deep=True).sum() / 1024**2:.0f} MB')
test.head()

Rows: 7,166,679
Columns: ['sensor_id', 'hour_timestamp', 'direction', 'hour', 'day_of_week', 'month', 'year', 'count']
Date range: 2019-08-01 00:00:00 → 2024-12-31 23:00:00
Sensors: 141
Memory: 968 MB


,sensor_id,hour_timestamp,direction,hour,day_of_week,month,year,count
0,1,2019-08-01 00:00:00,IN,0,Thursday,8,2019,0.0
1,1,2019-08-01 00:00:00,OUT,0,Thursday,8,2019,2.0
2,1,2019-08-01 01:00:00,IN,1,Thursday,8,2019,1.0
3,1,2019-08-01 01:00:00,OUT,1,Thursday,8,2019,0.0
4,1,2019-08-01 02:00:00,IN,2,Thursday,8,2019,1.0
